In [ ]:
!pip install redis

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 339.9/339.9 kB 4.3 MB/s eta 0:00:00


In [ ]:
import redis
from typing import Any, Dict, List, Optional

# Connect to local Redis (adjust host/port/password if needed)
"""Basic connection example.
"""


r = redis.Redis(
    host='redis-17263.c89.us-east-1-3.ec2.cloud.redislabs.com',
    port=17263,
    decode_responses=True,
    username="default",
    password="livFTRPNtHYZXrLwAGdX2ulAo3MGOhzN",
)

success = r.set('foo', 'bar')
# True

result = r.get('foo')
print(result)
# >>> bar



# Test connection
try:
    r.ping()
    print("✅ Connected to Redis!")
except redis.ConnectionError as e:
    print("❌ Redis connection failed:", e)

bar
✅ Connected to Redis!


In [ ]:
# -----------------
# Helper utilities
# -----------------
def _exists(r: redis.Redis, key: str) -> bool:
    return bool(r.exists(key))


def _to_int(x: Any) -> Optional[int]:
    try:
        return int(x)
    except Exception:
        return None


# =========== USER CRUD ===========
def add_user(user_id: int, name: str, email: str, region: str = "East") -> bool:
    uid = _to_int(user_id)
    if uid is None:
        raise ValueError("user_id must be int-like")

    key = f"user:{uid}"
    if _exists(r, key):
        return False

    pipe = r.pipeline()
    pipe.hset(key, mapping={"name": name, "email": email, "region": region})
    pipe.sadd("users", uid)
    pipe.execute()
    return True


def get_user(user_id: int) -> Optional[Dict[str, Any]]:
    uid = _to_int(user_id)
    if uid is None:
        raise ValueError("user_id must be int-like")

    key = f"user:{uid}"
    data = r.hgetall(key)
    if not data:
        return None
    return {"user_id": uid, **data}


def update_user_name(user_id: int, name: str) -> bool:
    uid = _to_int(user_id)
    key = f"user:{uid}"
    if not _exists(r, key):
        return False
    return bool(r.hset(key, "name", name))


def update_user_email(user_id: int, email: str) -> bool:
    uid = _to_int(user_id)
    key = f"user:{uid}"
    if not _exists(r, key):
        return False
    return bool(r.hset(key, "email", email))


def delete_user(user_id: int) -> bool:
    uid = _to_int(user_id)
    key = f"user:{uid}"
    if not _exists(r, key):
        return False

    order_set_key = f"orders_by_user:{uid}"
    order_ids = list(r.smembers(order_set_key))

    pipe = r.pipeline()
    for oid in order_ids:
        pipe.delete(f"order:{uid}:{oid}")
    pipe.delete(order_set_key)
    pipe.srem("users", uid)
    pipe.delete(key)
    pipe.execute()
    return True


def get_all_users() -> List[Dict[str, Any]]:
    ids = sorted([int(x) for x in r.smembers("users")])
    out: List[Dict[str, Any]] = []
    for uid in ids:
        u = get_user(uid)
        if u:
            out.append(u)
    return out


# =========== ORDER CRUD ===========
def add_order(user_id: int, order_id: int, product: str, amount: float) -> bool:
    uid = _to_int(user_id)
    oid = _to_int(order_id)
    if uid is None or oid is None:
        raise ValueError("user_id and order_id must be int-like")

    if not _exists(r, f"user:{uid}"):
        return False

    key = f"order:{uid}:{oid}"
    if _exists(r, key):
        return False

    pipe = r.pipeline()
    pipe.hset(key, mapping={"product": product, "amount": str(amount)})
    pipe.sadd(f"orders_by_user:{uid}", oid)
    pipe.execute()
    return True


def get_orders_by_user(user_id: int) -> List[Dict[str, Any]]:
    uid = _to_int(user_id)
    if uid is None:
        raise ValueError("user_id must be int-like")

    if not _exists(r, f"user:{uid}"):
        return []

    set_key = f"orders_by_user:{uid}"
    ids = sorted([int(x) for x in r.smembers(set_key)])
    out: List[Dict[str, Any]] = []
    for oid in ids:
        key = f"order:{uid}:{oid}"
        data = r.hgetall(key)
        if data:
            out.append(
                {
                    "user_id": uid,
                    "order_id": oid,
                    "product": data.get("product"),
                    "amount": float(data.get("amount", "0") or "0"),
                }
            )
    return out


def update_order_product(user_id: int, order_id: int, product: str) -> bool:
    uid = _to_int(user_id)
    oid = _to_int(order_id)
    key = f"order:{uid}:{oid}"
    if not _exists(r, key):
        return False
    return bool(r.hset(key, "product", product))


def update_order_amount(user_id: int, order_id: int, amount: float) -> bool:
    uid = _to_int(user_id)
    oid = _to_int(order_id)
    key = f"order:{uid}:{oid}"
    if not _exists(r, key):
        return False
    return bool(r.hset(key, "amount", str(amount)))


def delete_order(user_id: int, order_id: int) -> bool:
    uid = _to_int(user_id)
    oid = _to_int(order_id)
    key = f"order:{uid}:{oid}"
    if not _exists(r, key):
        return False

    pipe = r.pipeline()
    pipe.delete(key)
    pipe.srem(f"orders_by_user:{uid}", oid)
    pipe.execute()
    return True


# =========== QUERY HELPERS ===========
def get_orders_for_region(region: str) -> List[Dict[str, Any]]:
    out: List[Dict[str, Any]] = []
    for u in get_all_users():
        if u.get("region") == region:
            out.extend(get_orders_by_user(u["user_id"]))
    return out


def total_sales_for_user(user_id: int) -> float:
    return sum(o["amount"] for o in get_orders_by_user(user_id))


def orders_amount_gt(threshold: float) -> List[Dict[str, Any]]:
    out: List[Dict[str, Any]] = []
    for u in get_all_users():
        for o in get_orders_by_user(u["user_id"]):
            if o["amount"] > threshold:
                out.append(o)
    return out


# =========== PRODUCT CRUD ===========
def add_product(product_id: int, name: str, price: float) -> bool:
    pid = _to_int(product_id)
    if pid is None:
        raise ValueError("product_id must be int-like")

    key = f"product:{pid}"
    if _exists(r, key):
        return False

    pipe = r.pipeline()
    pipe.hset(key, mapping={"name": name, "price": str(price)})
    pipe.sadd("products", pid)
    pipe.execute()
    return True


def get_product(product_id: int) -> Optional[Dict[str, Any]]:
    pid = _to_int(product_id)
    if pid is None:
        raise ValueError("product_id must be int-like")

    key = f"product:{pid}"
    data = r.hgetall(key)
    if not data:
        return None

    return {
        "product_id": pid,
        "name": data.get("name"),
        "price": float(data.get("price", "0") or "0"),
    }


def get_all_products() -> List[Dict[str, Any]]:
    ids = sorted([int(x) for x in r.smembers("products")])
    out: List[Dict[str, Any]] = []
    for pid in ids:
        p = get_product(pid)
        if p:
            out.append(p)
    return out


def update_product_name(product_id: int, name: str) -> bool:
    pid = _to_int(product_id)
    key = f"product:{pid}"
    if not _exists(r, key):
        return False
    return bool(r.hset(key, "name", name))


def update_product_price(product_id: int, price: float) -> bool:
    pid = _to_int(product_id)
    key = f"product:{pid}"
    if not _exists(r, key):
        return False
    return bool(r.hset(key, "price", str(price)))


def delete_product(product_id: int) -> bool:
    pid = _to_int(product_id)
    key = f"product:{pid}"
    if not _exists(r, key):
        return False

    pipe = r.pipeline()
    pipe.delete(key)
    pipe.srem("products", pid)
    pipe.execute()
    return True

print("Functions defined.")

Functions defined.


In [ ]:
# Optional: clear existing DB (uncomment if you want a clean start)
# r.flushdb()

# Load USERS
add_user(1, "Alice",   "alice@example.com",   "East")
add_user(2, "Bob",     "bob@example.com",     "West")
add_user(3, "Charlie", "charlie@example.com", "East")
add_user(4, "David",   "david@example.com",   "West")
add_user(5, "Emma",    "emma@example.com",    "East")

# Load PRODUCTS
add_product(1, "Phone",      700)
add_product(2, "Console",    400)
add_product(3, "Headphones", 150)
add_product(4, "Keyboard",   120)
add_product(5, "Tablet",     500)
add_product(6, "Monitor",    250)
add_product(7, "Mouse",       50)
add_product(8, "Smartwatch", 300)

# Load ORDERS
add_order(1, 101, "Phone",      700)
add_order(2, 102, "Console",    400)
add_order(1, 103, "Headphones", 150)
add_order(3, 104, "Keyboard",   120)
add_order(2, 105, "Tablet",     500)
add_order(3, 106, "Monitor",    250)
add_order(3, 107, "Mouse",       50)
add_order(1, 108, "Smartwatch", 300)

print("Sample data loaded.")

Sample data loaded.


In [ ]:
# 1. Retrieve user 1 and write down their name and email.
u1 = get_user(1)
print("1) User 1:", u1["name"], u1["email"])

# 2. Retrieve all users and write down how many users the system contains.
users = get_all_users()
print("2) Number of users:", len(users))

# 3. Retrieve all orders belonging to user 1 and list the product names.
orders_u1 = get_orders_by_user(1)
print("3) User 1 order products:", [o["product"] for o in orders_u1])

# 4. Retrieve all products and write down all product names.
products = get_all_products()
print("4) All product names:", [p["name"] for p in products])

# 5. Retrieve product 3 and write down its product name and price.
p3 = get_product(3)
print("5) Product 3:", p3["name"], p3["price"])

# 6. Retrieve the total sales amount for user 1 and record the value.
total_u1 = total_sales_for_user(1)
print("6) Total sales for user 1:", total_u1)

# 7. Retrieve all orders whose amount is greater than 500 and list their order IDs.
gt_500 = orders_amount_gt(500)
print("7) Orders with amount > 500 (order_ids):", [o["order_id"] for o in gt_500])

# 8. Retrieve all orders placed by users in the East region and list all order IDs.
east_orders = get_orders_for_region("East")
print("8) East-region order IDs:", [o["order_id"] for o in east_orders])

# 9. Update the email of user 1, then retrieve the user and record the updated email.
update_user_email(1, "alice_new@example.com")
u1_updated = get_user(1)
print("9) Updated email for user 1:", u1_updated["email"])

# 10. Update the name of product 2, then retrieve the product and write down the updated name.
update_product_name(2, "Game Console")
p2_updated = get_product(2)
print("10) Updated name for product 2:", p2_updated["name"])

## Tasks 11–20

In [ ]:
# 11. Retrieve all users, identify one user from the West region,
# then retrieve their orders and write down all product names.
users = get_all_users()
west_user = next(u for u in users if u["region"] == "West")
west_orders = get_orders_by_user(west_user["user_id"])
print("11) West user:", west_user["name"], "| Products:", [o["product"] for o in west_orders])

# 12. Retrieve the orders of any user of your choice,
# then retrieve the matching product information for each product name and compare their prices.
orders_u3 = get_orders_by_user(3)
comparisons = []
for o in orders_u3:
    # match by product name to product table
    prod = next((p for p in get_all_products() if p["name"] == o["product"]), None)
    comparisons.append((o["product"], o["amount"], prod["price"] if prod else None))
print("12) User 3 order vs product price (name, order_amount, product_price):")
for row in comparisons:
    print("   ", row)

# 13. Retrieve all products and all orders, then determine whether every product
# name in the orders exists in the product list.
product_names = {p["name"] for p in get_all_products()}
all_orders = []
for u in get_all_users():
    all_orders.extend(get_orders_by_user(u["user_id"]))
order_products = {o["product"] for o in all_orders}
all_exist = order_products.issubset(product_names)
print("13) Do all order product names exist in product list?", all_exist)

# 14. Retrieve the total sales of user 1 and user 2 and identify which user has higher total sales.
total_u1 = total_sales_for_user(1)
total_u2 = total_sales_for_user(2)
higher = "User 1" if total_u1 > total_u2 else "User 2" if total_u2 > total_u1 else "Tie"
print("14) Total U1:", total_u1, "Total U2:", total_u2, "Higher:", higher)

# 15. Increase the price of product 5, retrieve it again, and write down the updated price.
p5 = get_product(5)
update_product_price(5, p5["price"] + 50)
p5_new = get_product(5)
print("15) Updated price of product 5:", p5_new["price"])

# 16. Retrieve all orders above 300, then retrieve the associated users and write down their names.
orders_gt_300 = orders_amount_gt(300)
user_names_gt_300 = [get_user(o["user_id"])["name"] for o in orders_gt_300]
print("16) Users with orders > 300:", user_names_gt_300)

# 17. Retrieve product 1 and the orders of user 1, then compare the product price to the amounts in those orders.
p1 = get_product(1)
orders_u1 = get_orders_by_user(1)
print("17) Product 1 price:", p1["price"])
print("    User 1 order amounts:", [o["amount"] for o in orders_u1])

# 18. Delete product 7 and then retrieve all products to confirm whether product 7 appears.
delete_product(7)
prod_ids_after_del7 = [p["product_id"] for p in get_all_products()]
print("18) Product IDs after deleting 7:", prod_ids_after_del7)

# 19. Retrieve all users, select any user, update their name,
# then retrieve that user again and write down the updated name.
any_user = get_all_users()[0]
update_user_name(any_user["user_id"], any_user["name"] + " Jr.")
any_user_updated = get_user(any_user["user_id"])
print("19) Updated name for selected user:", any_user_updated["name"])

# 20. Retrieve all users from the East region,
# then retrieve total sales for each and write the totals.
east_users = [u for u in get_all_users() if u["region"] == "East"]
print("20) East users total sales:")
for u in east_users:
    print("   ", u["name"], "->", total_sales_for_user(u["user_id"]))

## Tasks 21–30

In [ ]:
# 21. Retrieve all orders above 0 and all products,
# then determine whether any order amount differs from the corresponding product price.
all_orders = []
for u in get_all_users():
    all_orders.extend(get_orders_by_user(u["user_id"]))
products = get_all_products()
prod_price_map = {p["name"]: p["price"] for p in products}
diff_orders = [
    (o["order_id"], o["product"], o["amount"], prod_price_map.get(o["product"]))
    for o in all_orders
    if prod_price_map.get(o["product"]) is not None and o["amount"] != prod_price_map[o["product"]]
]
print("21) Orders with amount != product price:", diff_orders)

# 22. Retrieve all orders above 0, choose any order,
# then retrieve the associated user and write down their name.
any_order = all_orders[0]
user_for_order = get_user(any_order["user_id"])
print("22) Any order:", any_order["order_id"], "belongs to user:", user_for_order["name"])

# 23. Retrieve all orders above 0 and determine manually which three have the highest amounts.
sorted_orders = sorted(all_orders, key=lambda o: o["amount"], reverse=True)
top3 = sorted_orders[:3]
print("23) Top 3 orders by amount (order_id, amount):", [(o["order_id"], o["amount"]) for o in top3])

# 24. Retrieve all products, identify the most expensive product,
# then determine whether any order uses that product.
max_product = max(get_all_products(), key=lambda p: p["price"])
used_in_orders = any(o["product"] == max_product["name"] for o in all_orders)
print("24) Most expensive product:", max_product["name"], "Used in orders?", used_in_orders)

# 25. Retrieve two different products and compare their prices.
p1 = get_product(1)
p2 = get_product(2)
comparison = "Product 1 is more expensive" if p1["price"] > p2["price"] else "Product 2 is more expensive" if p2["price"] > p1["price"] else "Same price"
print("25) Compare product 1 and 2 prices:", p1["price"], "vs", p2["price"], "->", comparison)

# 26. Add a new order for any user, then retrieve that user’s orders and write down the newly added order.
new_order_id = 200
add_order(1, new_order_id, "Phone", get_product(1)["price"])
orders_u1_after = get_orders_by_user(1)
print("26) New order added for user 1 (last order):", orders_u1_after[-1])

# 27. Retrieve user 3, delete user 3,
# then retrieve all users and orders and write down what changed.
u3_before = get_user(3)
orders_u3_before = get_orders_by_user(3)
delete_user(3)
users_after_del3 = get_all_users()
orders_after_del3 = []
for u in users_after_del3:
    orders_after_del3.extend(get_orders_by_user(u["user_id"]))
print("27) User 3 before delete:", u3_before)
print("    Orders of user 3 before delete:", orders_u3_before)
print("    Users after delete (ids):", [u["user_id"] for u in users_after_del3])
print("    Remaining order user_ids:", sorted({o["user_id"] for o in orders_after_del3}))

# 28. Retrieve all East-region orders and compare the product names with the product table.
east_orders = get_orders_for_region("East")
east_product_names = {o["product"] for o in east_orders}
product_names = {p["name"] for p in get_all_products()}
print("28) East-region order product names:", east_product_names)
print("    Subset of product table names?", east_product_names.issubset(product_names))

# 29. Update the price of product 1, add a new order using that product,
# then retrieve user 1’s orders and write down the new order amount.
p1_old = get_product(1)
update_product_price(1, p1_old["price"] + 25)
p1_new = get_product(1)
add_order(1, 201, "Phone", p1_new["price"])
orders_u1_now = get_orders_by_user(1)
print("29) New order for user 1 (last order):", orders_u1_now[-1])

# 30. Lower the price of product 6 and retrieve the product again to confirm the updated price.
p6_old = get_product(6)
update_product_price(6, p6_old["price"] - 25)
p6_new = get_product(6)
print("30) Updated price for product 6:", p6_new["price"])

## Tasks 31–40

In [ ]:
# 31. Retrieve all users and determine which user has the longest name.
users = get_all_users()
longest_user = max(users, key=lambda u: len(u["name"]))
print("31) User with longest name:", longest_user["name"])

# 32. Retrieve all orders above 0 and identify which product name appears most frequently.
all_orders = []
for u in get_all_users():
    all_orders.extend(get_orders_by_user(u["user_id"]))
from collections import Counter
prod_counts = Counter(o["product"] for o in all_orders)
most_common_product, freq = prod_counts.most_common(1)[0]
print("32) Most frequent product in orders:", most_common_product, "(", freq, "times )")

# 33. Update both the name and email of user 2,
# then retrieve user 2 and write down both updated fields.
update_user_name(2, "Bob Updated")
update_user_email(2, "bob_updated@example.com")
u2_updated = get_user(2)
print("33) Updated user 2:", u2_updated["name"], u2_updated["email"])

# 34. Retrieve all orders of user 1 and compute the average order amount manually from the printed values.
orders_u1 = get_orders_by_user(1)
amounts_u1 = [o["amount"] for o in orders_u1]
avg_u1 = sum(amounts_u1) / len(amounts_u1) if amounts_u1 else 0
print("34) User 1 order amounts:", amounts_u1)
print("    Average order amount:", avg_u1)

# 35. Retrieve all products and determine which one has the smallest price.
products = get_all_products()
cheapest = min(products, key=lambda p: p["price"])
print("35) Cheapest product:", cheapest["name"], "price:", cheapest["price"])

# 36. Retrieve all orders above 300 and compare them to the product table
# to identify which orders correspond to expensive products.
orders_gt_300 = [o for o in all_orders if o["amount"] > 300]
expensive_info = [
    (o["order_id"], o["product"], o["amount"])
    for o in orders_gt_300
]
print("36) Orders with amount > 300:", expensive_info)

# 37. Add a new product, retrieve all products, and determine where the new product fits in the price ranking.
add_product(9, "Speaker", 275)
products = get_all_products()
products_sorted = sorted(products, key=lambda p: p["price"])
print("37) Products sorted by price (id, name, price):")
for p in products_sorted:
    print("   ", p["product_id"], p["name"], p["price"])

# 38. Retrieve all users and all orders, then determine which user has the greatest number of orders.
orders_by_user_count = {}
for u in get_all_users():
    cnt = len(get_orders_by_user(u["user_id"]))
    orders_by_user_count[u["name"]] = cnt
max_user = max(orders_by_user_count.items(), key=lambda kv: kv[1])
print("38) User with most orders:", max_user[0], "with", max_user[1], "orders")

# 39. Update the prices of products 1 and 2,
# then retrieve all products and confirm the updated prices.
update_product_price(1, get_product(1)["price"] + 10)
update_product_price(2, get_product(2)["price"] + 20)
p1_now = get_product(1)
p2_now = get_product(2)
print("39) New price for product 1:", p1_now["price"])
print("    New price for product 2:", p2_now["price"])

# 40. Retrieve all orders above 0 and all products,
# then determine whether any product has never been ordered.
all_orders = []
for u in get_all_users():
    all_orders.extend(get_orders_by_user(u["user_id"]))
ordered_product_names = {o["product"] for o in all_orders}
all_product_names = {p["name"] for p in get_all_products()}
never_ordered = all_product_names - ordered_product_names
print("40) Products never ordered:", never_ordered)

## Tasks 41–50

In [ ]:
# 41. Retrieve all users and write down the name of the first user in the list.
users = get_all_users()
print("41) First user name:", users[0]["name"])

# 42. Retrieve all users and write down the email of the second user in the list.
if len(users) > 1:
    print("42) Second user email:", users[1]["email"])
else:
    print("42) Less than 2 users in system.")

# 43. Retrieve all orders of user 1 and write down the product name of the first order.
orders_u1 = get_orders_by_user(1)
if orders_u1:
    print("43) First order product for user 1:", orders_u1[0]["product"])
else:
    print("43) User 1 has no orders.")

# 44. Retrieve all orders of user 1 and write down the amount of the last order.
if orders_u1:
    print("44) Last order amount for user 1:", orders_u1[-1]["amount"])
else:
    print("44) User 1 has no orders.")

# 45. Retrieve all products and write down the name of the third product.
products = get_all_products()
if len(products) >= 3:
    print("45) Third product name:", products[2]["name"])
else:
    print("45) Less than 3 products.")

# 46. Retrieve user 1 and write down this user’s email only.
u1 = get_user(1)
print("46) User 1 email:", u1["email"])

# 47. Retrieve product 1 and write down this product’s price only.
p1 = get_product(1)
print("47) Product 1 price:", p1["price"])

# 48. Retrieve all East-region orders and write down the user ID of the first returned order.
east_orders = get_orders_for_region("East")
if east_orders:
    print("48) User ID of first East-region order:", east_orders[0]["user_id"])
else:
    print("48) No East-region orders.")

# 49. Retrieve all orders above 500 and write down the product name of the first returned order.
orders_gt_500 = orders_amount_gt(500)
if orders_gt_500:
    print("49) Product of first order > 500:", orders_gt_500[0]["product"])
else:
    print("49) No orders > 500.")

# 50. Retrieve all users and all orders for user 1
# and write down a pair consisting of the name of the first user
# and the product of the first order of user 1.
users = get_all_users()
orders_u1 = get_orders_by_user(1)
if users and orders_u1:
    pair = (users[0]["name"], orders_u1[0]["product"])
    print("50) (First user name, first product of user 1):", pair)
else:
    print("50) Not enough data for pair.")

## Export Redis database to JSON

In [ ]:
import json

def export_redis_to_json(filename: str = "redis_dump.json") -> None:
    out = {}
    for key in r.scan_iter(match='*', count=1000):
        t = r.type(key)
        if t == 'string':
            out[key] = r.get(key)
        elif t == 'hash':
            out[key] = r.hgetall(key)
        elif t == 'list':
            out[key] = r.lrange(key, 0, -1)
        elif t == 'set':
            out[key] = list(r.smembers(key))
        elif t == 'zset':
            out[key] = r.zrange(key, 0, -1, withscores=True)
        else:
            out[key] = {'_type': t}
    with open(filename, 'w', encoding='utf-8') as f:
        json.dump(out, f, ensure_ascii=False, indent=2)
    print(f"✅ Exported Redis data to {filename}")

# Run export
export_redis_to_json()

# Task
Create a new code cell for Task 1, extracting the relevant code from cell `0c46ea0b`, to retrieve user 1's name and email.

## Process Tasks 1-10

### Subtask:
Extract Task 1 from cell `0c46ea0b` and create a new code cell for it.


**Reasoning**:
The subtask requires extracting Task 1 from the existing cell and placing it into a new code cell. This command will create a new code cell containing the code for Task 1.



In [1]:
u1 = get_user(1)
print("1) User 1:", u1["name"], u1["email"])

NameError: name 'get_user' is not defined

**Reasoning**:
The `NameError` occurs because the `get_user` function, and its dependencies like the Redis connection `r`, were not defined in the current kernel session. To fix this, the cells defining these functions and the connection (`64a576e3` and `472de38b`) must be executed. As per the guidelines, I will provide a new code cell that consolidates these necessary definitions to ensure the `get_user` function is available for subsequent tasks.



In [2]:
import redis
from typing import Any, Dict, List, Optional

# Connect to local Redis (adjust host/port/password if needed)
"""Basic connection example.
"""

r = redis.Redis(
    host='redis-17263.c89.us-east-1-3.ec2.cloud.redislabs.com',
    port=17263,
    decode_responses=True,
    username="default",
    password="livFTRPNtHYZXrLwAGdX2ulAo3MGOhzN",
)

# Test connection
try:
    r.ping()
    print("✅ Connected to Redis!")
except redis.ConnectionError as e:
    print("❌ Redis connection failed:", e)

# -----------------
# Helper utilities
# -----------------
def _exists(r: redis.Redis, key: str) -> bool:
    return bool(r.exists(key))


def _to_int(x: Any) -> Optional[int]:
    try:
        return int(x)
    except Exception:
        return None


# =========== USER CRUD ===========
def add_user(user_id: int, name: str, email: str, region: str = "East") -> bool:
    uid = _to_int(user_id)
    if uid is None:
        raise ValueError("user_id must be int-like")

    key = f"user:{uid}"
    if _exists(r, key):
        return False

    pipe = r.pipeline()
    pipe.hset(key, mapping={"name": name, "email": email, "region": region})
    pipe.sadd("users", uid)
    pipe.execute()
    return True


def get_user(user_id: int) -> Optional[Dict[str, Any]]:
    uid = _to_int(user_id)
    if uid is None:
        raise ValueError("user_id must be int-like")

    key = f"user:{uid}"
    data = r.hgetall(key)
    if not data:
        return None
    return {"user_id": uid, **data}


def update_user_name(user_id: int, name: str) -> bool:
    uid = _to_int(user_id)
    key = f"user:{uid}"
    if not _exists(r, key):
        return False
    return bool(r.hset(key, "name", name))


def update_user_email(user_id: int, email: str) -> bool:
    uid = _to_int(user_id)
    key = f"user:{uid}"
    if not _exists(r, key):
        return False
    return bool(r.hset(key, "email", email))


def delete_user(user_id: int) -> bool:
    uid = _to_int(user_id)
    key = f"user:{uid}"
    if not _exists(r, key):
        return False

    order_set_key = f"orders_by_user:{uid}"
    order_ids = list(r.smembers(order_set_key))

    pipe = r.pipeline()
    for oid in order_ids:
        pipe.delete(f"order:{uid}:{oid}")
    pipe.delete(order_set_key)
    pipe.srem("users", uid)
    pipe.delete(key)
    pipe.execute()
    return True


def get_all_users() -> List[Dict[str, Any]]:
    ids = sorted([int(x) for x in r.smembers("users")])
    out: List[Dict[str, Any]] = []
    for uid in ids:
        u = get_user(uid)
        if u:
            out.append(u)
    return out


# =========== ORDER CRUD ===========
def add_order(user_id: int, order_id: int, product: str, amount: float) -> bool:
    uid = _to_int(user_id)
    oid = _to_int(order_id)
    if uid is None or oid is None:
        raise ValueError("user_id and order_id must be int-like")

    if not _exists(r, f"user:{uid}"):
        return False

    key = f"order:{uid}:{oid}"
    if _exists(r, key):
        return False

    pipe = r.pipeline()
    pipe.hset(key, mapping={"product": product, "amount": str(amount)})
    pipe.sadd(f"orders_by_user:{uid}", oid)
    pipe.execute()
    return True


def get_orders_by_user(user_id: int) -> List[Dict[str, Any]]:
    uid = _to_int(user_id)
    if uid is None:
        raise ValueError("user_id must be int-like")

    if not _exists(r, f"user:{uid}"):
        return []

    set_key = f"orders_by_user:{uid}"
    ids = sorted([int(x) for x in r.smembers(set_key)])
    out: List[Dict[str, Any]] = []
    for oid in ids:
        key = f"order:{uid}:{oid}"
        data = r.hgetall(key)
        if data:
            out.append(
                {
                    "user_id": uid,
                    "order_id": oid,
                    "product": data.get("product"),
                    "amount": float(data.get("amount", "0") or "0"),
                }
            )
    return out


def update_order_product(user_id: int, order_id: int, product: str) -> bool:
    uid = _to_int(user_id)
    oid = _to_int(order_id)
    key = f"order:{uid}:{oid}"
    if not _exists(r, key):
        return False
    return bool(r.hset(key, "product", product))


def update_order_amount(user_id: int, order_id: int, amount: float) -> bool:
    uid = _to_int(user_id)
    oid = _to_int(order_id)
    key = f"order:{uid}:{oid}"
    if not _exists(r, key):
        return False
    return bool(r.hset(key, "amount", str(amount)))


def delete_order(user_id: int, order_id: int) -> bool:
    uid = _to_int(user_id)
    oid = _to_int(order_id)
    key = f"order:{uid}:{oid}"
    if not _exists(r, key):
        return False

    pipe = r.pipeline()
    pipe.delete(key)
    pipe.srem(f"orders_by_user:{uid}", oid)
    pipe.execute()
    return True


# =========== QUERY HELPERS ===========
def get_orders_for_region(region: str) -> List[Dict[str, Any]]:
    out: List[Dict[str, Any]] = []
    for u in get_all_users():
        if u.get("region") == region:
            out.extend(get_orders_by_user(u["user_id"]))
    return out


def total_sales_for_user(user_id: int) -> float:
    return sum(o["amount"] for o in get_orders_by_user(user_id))


def orders_amount_gt(threshold: float) -> List[Dict[str, Any]]:
    out: List[Dict[str, Any]] = []
    for u in get_all_users():
        for o in get_orders_by_user(u["user_id"]):
            if o["amount"] > threshold:
                out.append(o)
    return out


# =========== PRODUCT CRUD ===========
def add_product(product_id: int, name: str, price: float) -> bool:
    pid = _to_int(product_id)
    if pid is None:
        raise ValueError("product_id must be int-like")

    key = f"product:{pid}"
    if _exists(r, key):
        return False

    pipe = r.pipeline()
    pipe.hset(key, mapping={"name": name, "price": str(price)})
    pipe.sadd("products", pid)
    pipe.execute()
    return True


def get_product(product_id: int) -> Optional[Dict[str, Any]]:
    pid = _to_int(product_id)
    if pid is None:
        raise ValueError("product_id must be int-like")

    key = f"product:{pid}"
    data = r.hgetall(key)
    if not data:
        return None

    return {
        "product_id": pid,
        "name": data.get("name"),
        "price": float(data.get("price", "0") or "0"),
    }


def get_all_products() -> List[Dict[str, Any]]:
    ids = sorted([int(x) for x in r.smembers("products")])
    out: List[Dict[str, Any]] = []
    for pid in ids:
        p = get_product(pid)
        if p:
            out.append(p)
    return out


def update_product_name(product_id: int, name: str) -> bool:
    pid = _to_int(product_id)
    key = f"product:{pid}"
    if not _exists(r, key):
        return False
    return bool(r.hset(key, "name", name))


def update_product_price(product_id: int, price: float) -> bool:
    pid = _to_int(product_id)
    key = f"product:{pid}"
    if not _exists(r, key):
        return False
    return bool(r.hset(key, "price", str(price)))


def delete_product(product_id: int) -> bool:
    pid = _to_int(product_id)
    key = f"product:{pid}"
    if not _exists(r, key):
        return False

    pipe = r.pipeline()
    pipe.delete(key)
    pipe.srem("products", pid)
    pipe.execute()
    return True

print("Redis connection and functions defined.")

ModuleNotFoundError: No module named 'redis'

**Reasoning**:
The previous attempt to consolidate the Redis connection and functions failed with a `ModuleNotFoundError` because the `redis` library was not available in the current kernel session. To resolve this, I need to ensure the `redis` library is installed by executing `!pip install redis` in a new cell before defining the connection and functions.



In [3]:
!pip install redis

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 354.2/354.2 kB 9.2 MB/s eta 0:00:00


**Reasoning**:
Since the `redis` library has now been successfully installed, I need to re-execute the cell that defines the Redis connection and all the helper functions (like `get_user`) to make them available in the current kernel session. This will resolve the `NameError` encountered earlier.



In [4]:
import redis
from typing import Any, Dict, List, Optional

# Connect to local Redis (adjust host/port/password if needed)
"""Basic connection example.
"""

r = redis.Redis(
    host='redis-17263.c89.us-east-1-3.ec2.cloud.redislabs.com',
    port=17263,
    decode_responses=True,
    username="default",
    password="livFTRPNtHYZXrLwAGdX2ulAo3MGOhzN",
)

# Test connection
try:
    r.ping()
    print("✅ Connected to Redis!")
except redis.ConnectionError as e:
    print("❌ Redis connection failed:", e)

# -----------------
# Helper utilities
# -----------------
def _exists(r: redis.Redis, key: str) -> bool:
    return bool(r.exists(key))


def _to_int(x: Any) -> Optional[int]:
    try:
        return int(x)
    except Exception:
        return None


# =========== USER CRUD ===========
def add_user(user_id: int, name: str, email: str, region: str = "East") -> bool:
    uid = _to_int(user_id)
    if uid is None:
        raise ValueError("user_id must be int-like")

    key = f"user:{uid}"
    if _exists(r, key):
        return False

    pipe = r.pipeline()
    pipe.hset(key, mapping={"name": name, "email": email, "region": region})
    pipe.sadd("users", uid)
    pipe.execute()
    return True


def get_user(user_id: int) -> Optional[Dict[str, Any]]:
    uid = _to_int(user_id)
    if uid is None:
        raise ValueError("user_id must be int-like")

    key = f"user:{uid}"
    data = r.hgetall(key)
    if not data:
        return None
    return {"user_id": uid, **data}


def update_user_name(user_id: int, name: str) -> bool:
    uid = _to_int(user_id)
    key = f"user:{uid}"
    if not _exists(r, key):
        return False
    return bool(r.hset(key, "name", name))


def update_user_email(user_id: int, email: str) -> bool:
    uid = _to_int(user_id)
    key = f"user:{uid}"
    if not _exists(r, key):
        return False
    return bool(r.hset(key, "email", email))


def delete_user(user_id: int) -> bool:
    uid = _to_int(user_id)
    key = f"user:{uid}"
    if not _exists(r, key):
        return False

    order_set_key = f"orders_by_user:{uid}"
    order_ids = list(r.smembers(order_set_key))

    pipe = r.pipeline()
    for oid in order_ids:
        pipe.delete(f"order:{uid}:{oid}")
    pipe.delete(order_set_key)
    pipe.srem("users", uid)
    pipe.delete(key)
    pipe.execute()
    return True


def get_all_users() -> List[Dict[str, Any]]:
    ids = sorted([int(x) for x in r.smembers("users")])
    out: List[Dict[str, Any]] = []
    for uid in ids:
        u = get_user(uid)
        if u:
            out.append(u)
    return out


# =========== ORDER CRUD ===========
def add_order(user_id: int, order_id: int, product: str, amount: float) -> bool:
    uid = _to_int(user_id)
    oid = _to_int(order_id)
    if uid is None or oid is None:
        raise ValueError("user_id and order_id must be int-like")

    if not _exists(r, f"user:{uid}"):
        return False

    key = f"order:{uid}:{oid}"
    if _exists(r, key):
        return False

    pipe = r.pipeline()
    pipe.hset(key, mapping={"product": product, "amount": str(amount)})
    pipe.sadd(f"orders_by_user:{uid}", oid)
    pipe.execute()
    return True


def get_orders_by_user(user_id: int) -> List[Dict[str, Any]]:
    uid = _to_int(user_id)
    if uid is None:
        raise ValueError("user_id must be int-like")

    if not _exists(r, f"user:{uid}"):
        return []

    set_key = f"orders_by_user:{uid}"
    ids = sorted([int(x) for x in r.smembers(set_key)])
    out: List[Dict[str, Any]] = []
    for oid in ids:
        key = f"order:{uid}:{oid}"
        data = r.hgetall(key)
        if data:
            out.append(
                {
                    "user_id": uid,
                    "order_id": oid,
                    "product": data.get("product"),
                    "amount": float(data.get("amount", "0") or "0"),
                }
            )
    return out


def update_order_product(user_id: int, order_id: int, product: str) -> bool:
    uid = _to_int(user_id)
    oid = _to_int(order_id)
    key = f"order:{uid}:{oid}"
    if not _exists(r, key):
        return False
    return bool(r.hset(key, "product", product))


def update_order_amount(user_id: int, order_id: int, amount: float) -> bool:
    uid = _to_int(user_id)
    oid = _to_int(order_id)
    key = f"order:{uid}:{oid}"
    if not _exists(r, key):
        return False
    return bool(r.hset(key, "amount", str(amount)))


def delete_order(user_id: int, order_id: int) -> bool:
    uid = _to_int(user_id)
    oid = _to_int(order_id)
    key = f"order:{uid}:{oid}"
    if not _exists(r, key):
        return False

    pipe = r.pipeline()
    pipe.delete(key)
    pipe.srem(f"orders_by_user:{uid}", oid)
    pipe.execute()
    return True


# =========== QUERY HELPERS ===========
def get_orders_for_region(region: str) -> List[Dict[str, Any]]:
    out: List[Dict[str, Any]] = []
    for u in get_all_users():
        if u.get("region") == region:
            out.extend(get_orders_by_user(u["user_id"]))
    return out


def total_sales_for_user(user_id: int) -> float:
    return sum(o["amount"] for o in get_orders_by_user(user_id))


def orders_amount_gt(threshold: float) -> List[Dict[str, Any]]:
    out: List[Dict[str, Any]] = []
    for u in get_all_users():
        for o in get_orders_by_user(u["user_id"]):
            if o["amount"] > threshold:
                out.append(o)
    return out


# =========== PRODUCT CRUD ===========
def add_product(product_id: int, name: str, price: float) -> bool:
    pid = _to_int(product_id)
    if pid is None:
        raise ValueError("product_id must be int-like")

    key = f"product:{pid}"
    if _exists(r, key):
        return False

    pipe = r.pipeline()
    pipe.hset(key, mapping={"name": name, "price": str(price)})
    pipe.sadd("products", pid)
    pipe.execute()
    return True


def get_product(product_id: int) -> Optional[Dict[str, Any]]:
    pid = _to_int(product_id)
    if pid is None:
        raise ValueError("product_id must be int-like")

    key = f"product:{pid}"
    data = r.hgetall(key)
    if not data:
        return None

    return {
        "product_id": pid,
        "name": data.get("name"),
        "price": float(data.get("price", "0") or "0"),
    }


def get_all_products() -> List[Dict[str, Any]]:
    ids = sorted([int(x) for x in r.smembers("products")])
    out: List[Dict[str, Any]] = []
    for pid in ids:
        p = get_product(pid)
        if p:
            out.append(p)
    return out


def update_product_name(product_id: int, name: str) -> bool:
    pid = _to_int(product_id)
    key = f"product:{pid}"
    if not _exists(r, key):
        return False
    return bool(r.hset(key, "name", name))


def update_product_price(product_id: int, price: float) -> bool:
    pid = _to_int(product_id)
    key = f"product:{pid}"
    if not _exists(r, key):
        return False
    return bool(r.hset(key, "price", str(price)))


def delete_product(product_id: int) -> bool:
    pid = _to_int(product_id)
    key = f"product:{pid}"
    if not _exists(r, key):
        return False

    pipe = r.pipeline()
    pipe.delete(key)
    pipe.srem("products", pid)
    pipe.execute()
    return True

print("Redis connection and functions defined.")

✅ Connected to Redis!
Redis connection and functions defined.


**Reasoning**:
With the `redis` library installed and all helper functions, including `get_user`, successfully defined and connected to Redis, I can now proceed to execute the original Task 1 code. This will retrieve user 1's name and email as required by the subtask.



In [6]:
u1 = get_user(1)
print("1) User 1:", u1["name"], u1["email"])

1) User 1: Alice new_email@example.com


## Extract Task 2

### Subtask:
Extract Task 2 from cell `0c46ea0b` and create a new code cell for it.


**Reasoning**:
To complete the subtask, I will extract Task 2's code from cell `0c46ea0b` and place it in a new code cell.

